<div style='background:#0f172a;padding:28px 32px;border-radius:12px;color:#e2e8f0;font-family:system-ui,Arial'>
<h1 style='margin:0;font-size:22px'>EDA — BraTS 2024 GLI · análisis de intensidad para segmentación clásica</h1>
<p style='color:#94a3b8;margin:8px 0 0'>Versión rápida: <b>una sola pasada</b> sobre <b>150 casos</b> calcula histogramas, separabilidad por sub-región, Otsu/bimodalidad, variabilidad, semillas, geometría y volúmenes.<br>
Salidas en <code>final-project/eda/</code> para los notebooks de limpieza/registro y segmentación.<br>
Abel Albuez · Victoria Acero · Santiago Gil</p></div>

In [ ]:
# === Setup ===
import importlib, subprocess, sys
for pkg, mod in [('nibabel','nibabel')]:
    if importlib.util.find_spec(mod) is None:
        subprocess.run([sys.executable,'-m','pip','install',pkg,'--quiet'])

import os, glob, gc, json, io, base64, zipfile, shutil, time, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import nibabel as nib
from scipy import stats as spstats
from scipy import ndimage as ndi
from skimage.filters import threshold_multiotsu
import matplotlib
import matplotlib.pyplot as plt
print('Librerías listas.')

In [ ]:
# === Fuente de datos (Drive) — robusta ===
try:
    from google.colab import drive
    drive.mount('/content/drive'); EN_COLAB = True
except Exception:
    EN_COLAB = False
if EN_COLAB:
    try: os.listdir('/content/drive/MyDrive')
    except Exception:
        from google.colab import drive; drive.mount('/content/drive', force_remount=True)

# >>> AJUSTA SI HACE FALTA <<<
DRIVE_DIR   = '/content/drive/MyDrive/BRATS-2024'
COPIAR_ZIP_LOCAL = False   # False = lee solo los 150 casos del ZIP en Drive (no copia el archivo entero)

def _es_entrenamiento(z):
    n = os.path.basename(z).lower()
    return ('training' in n) and ('validation' not in n)
def _encontrar_zip(d):
    cands = sorted(glob.glob(os.path.join(d, '*.zip')))
    if not cands and os.path.isdir('/content/drive/MyDrive'):
        cands = sorted(glob.glob('/content/drive/MyDrive/**/*.zip', recursive=True))
    ent = [z for z in cands if _es_entrenamiento(z)]
    if ent:
        principal = [z for z in ent if 'additional' not in os.path.basename(z).lower()]
        return (principal or ent)[0]
    return cands[0] if cands else None

TRAINING_ZIP = _encontrar_zip(DRIVE_DIR)
print('Drive:', EN_COLAB, '| DRIVE_DIR existe:', os.path.isdir(DRIVE_DIR))
if os.path.isdir(DRIVE_DIR):
    print('Contenido:', [os.path.basename(p) for p in glob.glob(os.path.join(DRIVE_DIR,'*'))])
print('ZIP elegido:', TRAINING_ZIP)
if TRAINING_ZIP is None:
    raise FileNotFoundError('No se encontró un .zip de entrenamiento en Drive. Revisa DRIVE_DIR o el montaje.')

if TRAINING_ZIP and COPIAR_ZIP_LOCAL and TRAINING_ZIP.startswith('/content/drive'):
    _local = os.path.join('/content', os.path.basename(TRAINING_ZIP))
    try: _ok = os.path.exists(_local) and os.path.getsize(_local) == os.path.getsize(TRAINING_ZIP)
    except Exception: _ok = os.path.exists(_local)
    if not _ok:
        print('Copiando ZIP a disco local (una sola vez)... puede tardar unos minutos.')
        shutil.copy(TRAINING_ZIP, _local)
    TRAINING_ZIP = _local

EDA_DIR = os.path.join(DRIVE_DIR, 'final-project', 'eda') if os.path.isdir(DRIVE_DIR) else 'salidas_eda'
os.makedirs(EDA_DIR, exist_ok=True)
TMP_DIR = '/content/_eda_tmp'; os.makedirs(TMP_DIR, exist_ok=True)
print('Salidas del EDA ->', EDA_DIR)

In [ ]:
# === Configuración ===
N_CASOS   = 25                     # <<< casos del EDA (cámbialo aquí si hace falta)
RNG_SEED  = 42
MODALITIES_IMG = ['t1n','t1c','t2w','t2f']
MOD_LABELS = {'t1n':'T1n','t1c':'T1c','t2w':'T2w','t2f':'T2-FLAIR'}
LABEL_MAP  = {1:'NETC', 2:'SNFH', 3:'ET', 4:'RC'}     # BraTS 2024 GLI

# Muestreo de voxeles (acota memoria y tiempo; suficiente para histogramas suaves)
VOX_HIST   = 3000   # voxeles de cerebro por caso para el histograma global por modalidad
VOX_SUB    = 1500   # voxeles por caso y sub-región para separabilidad
VOX_OTSU   = 40000  # voxeles por caso para Otsu/bimodalidad
NBINS      = 120
print('N_CASOS =', N_CASOS)

In [ ]:
# === Utilidades ===
def downsample(arr, n, seed=RNG_SEED):
    arr = np.asarray(arr).ravel()
    if arr.size <= n: return arr
    return arr[np.random.default_rng(seed).choice(arr.size, n, replace=False)]

def coef_bimodalidad(x):
    x = np.asarray(x); n = x.size
    if n < 4: return np.nan
    g1 = spstats.skew(x); g2 = spstats.kurtosis(x)
    den = g2 + 3.0*((n-1)**2)/((n-2)*(n-3))
    return (g1**2 + 1.0)/den if den != 0 else np.nan

def hist_norm(x, bins):
    h, _ = np.histogram(np.asarray(x), bins=bins)
    s = h.sum(); return (h/s) if s else h.astype(float)
def ovl(h1, h2):  return float(np.minimum(h1, h2).sum())
def bhatt(h1, h2): return float(-np.log(np.sum(np.sqrt(h1*h2)) + 1e-12))

def centroide(mask):
    idx = np.argwhere(mask)
    if len(idx) == 0: return None
    return tuple(int(round(c)) for c in idx.mean(0))

def _miembros_por_caso(zf, case_ids):
    cset = set(case_ids); mp = {}
    for m in zf.namelist():
        for p in m.split('/'):
            if p in cset: mp.setdefault(p, []).append(m); break
    return mp

def _cargar(path):
    """Carga rápida: float32 directo (evita el float64 de get_fdata por defecto)."""
    im = nib.load(path)
    return im.get_fdata(dtype=np.float32), im

def fig_to_b64(fig):
    buf = io.BytesIO(); fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    plt.close(fig); return 'data:image/png;base64,' + base64.b64encode(buf.getvalue()).decode()

print('Utilidades listas.')

In [ ]:
# === Selección de 150 casos (barato: solo lee la lista del ZIP, no extrae) ===
with zipfile.ZipFile(TRAINING_ZIP) as zf:
    _segs = [n for n in zf.namelist() if n.endswith('-seg.nii.gz')]
    todos = sorted({os.path.basename(n)[:-len('-seg.nii.gz')] for n in _segs})
print('Casos con seg en el ZIP:', len(todos))
rng = np.random.default_rng(RNG_SEED)
CASOS = sorted(rng.choice(todos, size=min(N_CASOS, len(todos)), replace=False).tolist())
print('Casos seleccionados:', len(CASOS))

---
## Pasada única sobre los 150 casos

Por cada caso se extrae del ZIP, se cargan las 4 modalidades + `seg`, y en ese mismo momento se acumula **todo**: volúmenes de tumor (`df_vol`), histogramas por modalidad (A), separabilidad sub-región vs sano (B), Otsu/bimodalidad por caso (C), media/p99 por caso (D), intensidad de semilla en T1c (E) y geometría. Una sola lectura de cada volumen.

In [ ]:
# === PASADA ÚNICA (lo pesado: ~N x 5 lecturas) ===
t0 = time.time()
res_mod  = {m: [] for m in MODALITIES_IMG}                                   # A
res_sub  = {m: {ln: [] for ln in LABEL_MAP.values()} for m in MODALITIES_IMG}# B
res_sano = {m: [] for m in MODALITIES_IMG}                                   # B
filas_vol, filas_stats, filas_otsu, filas_seed, filas_geo = [], [], [], [], []
sin_archivos = 0

with zipfile.ZipFile(TRAINING_ZIP) as zf:
    mp = _miembros_por_caso(zf, CASOS)
    for k, cid in enumerate(CASOS, 1):
        # limpiar TMP y extraer solo los miembros de este caso
        shutil.rmtree(TMP_DIR, ignore_errors=True); os.makedirs(TMP_DIR, exist_ok=True)
        for mem in mp.get(cid, []):
            try: zf.extract(mem, TMP_DIR)
            except Exception: pass
        # localizar cada archivo por su NOMBRE real (robusto a la carpeta envolvente del ZIP)
        paths = {mod: (glob.glob(os.path.join(TMP_DIR, '**', f'{cid}-{mod}.nii*'), recursive=True) or [None])[0]
                 for mod in MODALITIES_IMG + ['seg']}
        if not any(paths.values()):
            sin_archivos += 1
            if k <= 3:
                print('  (aviso) sin archivos para', cid, '| miembros en ZIP:', len(mp.get(cid, [])))
            continue

        # --- seg + df_vol ---
        seg = None
        if paths.get('seg'):
            seg = np.asarray(nib.load(paths['seg']).dataobj).astype(np.int16)
            fila = {'case_id': cid, 'TumorTotal': int((seg > 0).sum())}
            for lbl, nombre in LABEL_MAP.items(): fila[nombre] = int((seg == lbl).sum())
            filas_vol.append(fila)
        sano = (seg == 0) if seg is not None else None

        # --- modalidades ---
        geo_done = False; t1c_vol = None
        for mod in MODALITIES_IMG:
            if not paths.get(mod): continue
            im = nib.load(paths[mod]); vol = im.get_fdata(dtype=np.float32)
            brain = vol > 0; bv = vol[brain]
            if bv.size:
                res_mod[mod].append(downsample(bv, VOX_HIST))
                s = downsample(bv, VOX_OTSU)
                try: thr = threshold_multiotsu(s, classes=3).tolist()
                except Exception: thr = [float(np.percentile(s,50)), float(np.percentile(s,80))]
                filas_otsu.append({'case_id': cid, 'modalidad': mod,
                                   'otsu_low': float(thr[0]), 'otsu_high': float(thr[-1]),
                                   'bimodalidad': float(coef_bimodalidad(s))})
                filas_stats.append({'case_id': cid, 'modalidad': mod,
                                    'media': float(bv.mean()), 'p99': float(np.percentile(bv,99))})
            if seg is not None:
                for lbl, nombre in LABEL_MAP.items():
                    vv = vol[seg == lbl]
                    if vv.size: res_sub[mod][nombre].append(downsample(vv, VOX_SUB))
                if sano is not None:
                    vs = vol[sano & brain]
                    if vs.size: res_sano[mod].append(downsample(vs, VOX_SUB))
            if not geo_done:
                z = tuple(round(float(x),2) for x in im.header.get_zooms()[:3])
                iso = (max(z)-min(z) < 0.05) and (abs(z[0]-1.0) < 0.05)
                filas_geo.append({'case_id': cid, 'shape':'x'.join(map(str, im.shape[:3])),
                                  'spacing':'x'.join(f'{q:.2f}' for q in z),
                                  'orientacion': ''.join(nib.aff2axcodes(im.affine)), 'iso_1mm': bool(iso)})
                geo_done = True
            if mod == 't1c': t1c_vol = vol
            del vol

        # E: semilla = intensidad T1c en el centroide del tumor (ET si hay, si no todo)
        if seg is not None and t1c_vol is not None:
            m_et = (seg == 3); m_obj = m_et if m_et.any() else (seg > 0)
            c = centroide(m_obj)
            if c is not None:
                filas_seed.append({'case_id': cid, 'intensidad_t1c': float(t1c_vol[c]),
                                   'fuente': 'ET' if m_et.any() else 'tumor'})
        del t1c_vol, seg; gc.collect()
        if k % 25 == 0 or k == len(CASOS):
            print(f'  {k}/{len(CASOS)}  ({time.time()-t0:.0f}s)  casos con datos: {len(filas_vol)}')

shutil.rmtree(TMP_DIR, ignore_errors=True)
if sin_archivos:
    print(f'  AVISO: {sin_archivos} casos sin archivos localizados (revisa estructura del ZIP).')
print(f'Pasada única lista en {time.time()-t0:.0f}s | casos con datos: {len(filas_vol)}')

In [ ]:
# === df_vol, geometría y selección de casos demostrativos ===
df_vol = pd.DataFrame(filas_vol)
if df_vol.empty or 'TumorTotal' not in df_vol.columns:
    raise RuntimeError('df_vol vacío: la pasada única no encontró segmentaciones. '
                       'Revisa el AVISO de la celda anterior (estructura del ZIP / extracción).')
df_vol.to_csv(os.path.join(EDA_DIR, 'EDA_volumenes_completo.csv'), index=False)
df_geo = pd.DataFrame(filas_geo)
df_geo.to_csv(os.path.join(EDA_DIR, 'geometria_casos.csv'), index=False)

dv = df_vol[df_vol['TumorTotal'] > 0].copy()
sel = {}
def _pick(cond, etiqueta, criterio):
    cand = dv[cond]
    if len(cand) == 0: return
    sel.setdefault(criterio(cand), []).append(etiqueta)
med = dv['TumorTotal'].median()
_pick(pd.Series(True, index=dv.index), 'Típico (mediana)',
      lambda c: c.iloc[(c['TumorTotal']-med).abs().argsort()].iloc[0]['case_id'])
_pick(pd.Series(True, index=dv.index), 'Tumor grande (~p95)',
      lambda c: c.sort_values('TumorTotal').iloc[min(len(c)-1, int(len(c)*0.95))]['case_id'])
_pick(pd.Series(True, index=dv.index), 'Tumor pequeño (~p5)',
      lambda c: c.sort_values('TumorTotal').iloc[int(len(c)*0.05)]['case_id'])
_pick(dv['RC'] > 0, 'Post-resección (RC>0)',
      lambda c: c.sort_values('RC', ascending=False).iloc[0]['case_id'])
_pick((dv['ET'] == 0) & (dv['SNFH'] > 0), 'Solo edema (sin ET)',
      lambda c: c.sort_values('SNFH', ascending=False).iloc[0]['case_id'])
rows = []
for cid, etqs in sel.items():
    r = df_vol[df_vol['case_id'] == cid].iloc[0]
    rows.append({'case_id': cid, 'motivo': ' / '.join(etqs), 'TumorTotal': int(r['TumorTotal']),
                 **{ln: int(r[ln]) for ln in LABEL_MAP.values()}})
df_demo = pd.DataFrame(rows)
df_demo.to_csv(os.path.join(EDA_DIR, 'casos_demostrativos.csv'), index=False)
print('df_vol:', len(df_vol), '| geometría:', len(df_geo), '| demostrativos:', len(df_demo))
display(df_demo)

In [ ]:
# === A — Histograma de intensidad por modalidad ===
acc_mod = {m: (np.concatenate(res_mod[m]) if res_mod[m] else np.array([])) for m in MODALITIES_IMG}
bins_mod = {}
fig, ax = plt.subplots(figsize=(9,4.5))
for m in MODALITIES_IMG:
    a = acc_mod[m]
    if a.size == 0: continue
    lo, hi = np.percentile(a, [0.5, 99.5]); bins_mod[m] = np.linspace(lo, hi, NBINS+1)
    h = hist_norm(a, bins_mod[m]); ctr = (bins_mod[m][:-1]+bins_mod[m][1:])/2
    ax.plot(ctr, h, label=MOD_LABELS[m], linewidth=1.8)
ax.set_xlabel('Intensidad (voxeles de cerebro)'); ax.set_ylabel('Densidad')
ax.set_title('A — Distribución de intensidad por modalidad'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); figA = fig_to_b64(fig)
print('Figura A lista.')

In [ ]:
# === B — Separabilidad sub-región vs tejido sano ===
ovl_rows, bh_rows = {}, {}
for m in MODALITIES_IMG:
    if m not in bins_mod: continue
    hs = hist_norm(np.concatenate(res_sano[m]), bins_mod[m]) if res_sano[m] else None
    ovl_rows[MOD_LABELS[m]] = {}; bh_rows[MOD_LABELS[m]] = {}
    for ln in LABEL_MAP.values():
        chunks = res_sub[m][ln]
        if chunks and hs is not None:
            hsub = hist_norm(np.concatenate(chunks), bins_mod[m])
            ovl_rows[MOD_LABELS[m]][ln] = round(ovl(hsub, hs), 3)
            bh_rows[MOD_LABELS[m]][ln]  = round(bhatt(hsub, hs), 3)
df_ovl = pd.DataFrame(ovl_rows)   # filas = sub-región, columnas = modalidad
df_bh  = pd.DataFrame(bh_rows)

# figura: distribuciones por sub-región en la modalidad de mejor separabilidad de ET
mod_fija = (df_ovl.loc['ET'].astype(float).idxmin() if 'ET' in df_ovl.index and len(df_ovl) else 'T1c')
mfk = next((k for k,v in MOD_LABELS.items() if v == mod_fija), 't1c')
fig, ax = plt.subplots(figsize=(9,4.5))
if mfk in bins_mod:
    ctr = (bins_mod[mfk][:-1]+bins_mod[mfk][1:])/2
    if res_sano[mfk]:
        ax.plot(ctr, hist_norm(np.concatenate(res_sano[mfk]), bins_mod[mfk]), 'k--', label='Sano', linewidth=1.5)
    for ln in LABEL_MAP.values():
        if res_sub[mfk][ln]:
            ax.plot(ctr, hist_norm(np.concatenate(res_sub[mfk][ln]), bins_mod[mfk]), label=ln, linewidth=1.6)
ax.set_title(f'B — Sub-regiones vs sano en {mod_fija} (menor solapamiento = más separable)')
ax.set_xlabel('Intensidad'); ax.set_ylabel('Densidad'); ax.legend(); ax.grid(alpha=0.3)
plt.tight_layout(); figB = fig_to_b64(fig)
print('Modalidad de mejor separabilidad de ET:', mod_fija)
display(df_ovl); display(df_bh)

In [ ]:
# === C — Umbrales de Otsu y bimodalidad por modalidad ===
df_otsu_caso = pd.DataFrame(filas_otsu)
resumen_otsu = (df_otsu_caso.groupby('modalidad')
                .agg(otsu_low=('otsu_low','median'), otsu_high=('otsu_high','median'),
                     bimodalidad=('bimodalidad','median')).reset_index())
resumen_otsu['modalidad'] = resumen_otsu['modalidad'].map(MOD_LABELS).fillna(resumen_otsu['modalidad'])
df_otsu_caso.to_csv(os.path.join(EDA_DIR, 'EDA_intensidad_otsu.csv'), index=False)

fig, ax = plt.subplots(figsize=(8,4))
order = [m for m in MODALITIES_IMG if m in df_otsu_caso['modalidad'].unique()]
ax.boxplot([df_otsu_caso[df_otsu_caso['modalidad']==m]['otsu_high'] for m in order],
           labels=[MOD_LABELS[m] for m in order])
ax.set_title('C — Umbral de Otsu alto por modalidad (entre casos)'); ax.set_ylabel('Intensidad'); ax.grid(alpha=0.3)
plt.tight_layout(); figC = fig_to_b64(fig)
display(resumen_otsu.round(2))

In [ ]:
# === D — Variabilidad de intensidad entre casos ===
df_stats = pd.DataFrame(filas_stats)
df_stats.to_csv(os.path.join(EDA_DIR, 'EDA_intensidad_stats.csv'), index=False)
tab_var = (df_stats.groupby('modalidad').agg(media_mean=('media','mean'), media_std=('media','std'),
            p99_mean=('p99','mean'), p99_std=('p99','std')).reset_index())
tab_var['modalidad'] = tab_var['modalidad'].map(MOD_LABELS).fillna(tab_var['modalidad'])

fig, axes = plt.subplots(1,2, figsize=(12,4))
order = [m for m in MODALITIES_IMG if m in df_stats['modalidad'].unique()]
axes[0].boxplot([df_stats[df_stats['modalidad']==m]['media'] for m in order], labels=[MOD_LABELS[m] for m in order])
axes[0].set_title('Media por caso'); axes[0].grid(alpha=0.3)
axes[1].boxplot([df_stats[df_stats['modalidad']==m]['p99'] for m in order], labels=[MOD_LABELS[m] for m in order])
axes[1].set_title('p99 por caso'); axes[1].grid(alpha=0.3)
fig.suptitle('D — Variabilidad de intensidad entre casos'); plt.tight_layout(); figD = fig_to_b64(fig)
display(tab_var.round(2))

In [ ]:
# === E — Semilla para crecimiento de regiones (intensidad T1c en el centroide) ===
df_seed = pd.DataFrame(filas_seed)
figE = None
if len(df_seed):
    df_seed.to_csv(os.path.join(EDA_DIR, 'EDA_intensidad_seed.csv'), index=False)
    s = df_seed['intensidad_t1c']; p10,p50,p90 = np.percentile(s,[10,50,90]); tol = float(s.std())
    fig, ax = plt.subplots(figsize=(8,4))
    ax.hist(s, bins=25, color='#EF4444', alpha=0.85)
    for x,l in [(p10,'p10'),(p50,'mediana'),(p90,'p90')]:
        ax.axvline(x, color='k', ls='--'); ax.text(x, ax.get_ylim()[1]*0.9, l, rotation=90)
    ax.set_title('E — Intensidad de semilla candidata (T1c en el centroide del tumor)')
    ax.set_xlabel('Intensidad T1c'); ax.set_ylabel('Casos'); ax.grid(alpha=0.3)
    plt.tight_layout(); figE = fig_to_b64(fig)
    print(f'Semilla T1c -> p10={p10:.1f} mediana={p50:.1f} p90={p90:.1f} | tolerancia ~1σ = ±{tol:.1f}')
    display(df_seed.round(1).head(12))
else:
    print('Sin datos de semilla.')

In [ ]:
# === Geometría y parámetros de registro (parametros_registro.json) ===
pct_iso = 100*df_geo['iso_1mm'].mean() if len(df_geo) else 0.0
orient0 = df_geo['orientacion'].mode().iat[0] if len(df_geo) else 'RAS'
spacing_obj = df_geo['spacing'].mode().iat[0] if len(df_geo) else '1.00x1.00x1.00'
# En BraTS GLI las modalidades ya vienen coregistradas (mismo shape/spacing/orientación) -> registro demostrativo
intra_coreg = (df_geo['shape'].nunique() == 1) and (df_geo['spacing'].nunique() == 1)
necesita_intra = not bool(intra_coreg)

CONCLUSION = ('Las modalidades ya vienen coregistradas e isotrópicas; el coregistro intra-sujeto es redundante. '
              'El registro se usa como demostración de la técnica (perturbación rígida conocida -> recuperación).'
              if not necesita_intra else
              'Hay heterogeneidad geométrica entre casos/modalidades: conviene coregistro multimodal intra-sujeto.')

params = {'modalidad_fija_recomendada': str(mod_fija),
          'spacing_objetivo': spacing_obj, 'orientacion': orient0,
          'metrica_recomendada': 'Informacion mutua (Mattes) - multimodal',
          'transformacion': 'Rigida (Euler3D / VersorRigid3D)',
          'registro_intra_sujeto_necesario': bool(necesita_intra),
          'pct_casos_iso_1mm': round(pct_iso,1),
          'casos_demostrativos': df_demo['case_id'].tolist(),
          'conclusion': CONCLUSION}
with open(os.path.join(EDA_DIR, 'parametros_registro.json'), 'w', encoding='utf-8') as f:
    json.dump(params, f, ensure_ascii=False, indent=2)

df_reg = pd.DataFrame([('Modalidad fija recomendada', str(mod_fija)),
                       ('Spacing objetivo', spacing_obj), ('Orientación', orient0),
                       ('% casos isotrópicos 1mm', f'{pct_iso:.1f}%'),
                       ('¿Coregistro intra-sujeto necesario?', 'Sí' if necesita_intra else 'No'),
                       ('Conclusión', CONCLUSION)], columns=['Aspecto','Valor'])
print('parametros_registro.json guardado.')
display(df_reg)

In [ ]:
# === Reporte HTML (figuras + tablas) ===
def tabla(df, t): return f"<h3>{t}</h3>" + df.to_html(index=False, border=0)
secs = [('A — Intensidad por modalidad', figA, None),
        ('B — Separabilidad sub-región vs sano', figB, [('Solapamiento (OVL) — menor = más separable', df_ovl.reset_index().rename(columns={'index':'sub-region'})),
                                                        ('Bhattacharyya — mayor = más separable', df_bh.reset_index().rename(columns={'index':'sub-region'}))]),
        ('C — Otsu y bimodalidad', figC, [('Resumen Otsu (mediana)', resumen_otsu)]),
        ('D — Variabilidad entre casos', figD, [('Resumen variabilidad', tab_var)])]
if figE is not None: secs.append(('E — Semilla (T1c)', figE, None))
secs.append(('F — Casos demostrativos', None, [('Subconjunto representativo', df_demo)]))
secs.append(('G — Registro', None, [('Parámetros', df_reg), ('Geometría (muestra)', df_geo.head(12))]))

partes = ["<h1>EDA — BraTS 2024 GLI (" + str(len(CASOS)) + " casos)</h1>"]
for titulo, fig, tablas in secs:
    partes.append(f"<section><h2>{titulo}</h2>")
    if fig: partes.append(f"<img src='{fig}' style='max-width:760px'>")
    for tt, df in (tablas or []): partes.append(tabla(df.round(3) if df.select_dtypes('number').shape[1] else df, tt))
    partes.append("</section>")
html = "<html><head><meta charset='utf-8'><style>body{font-family:system-ui,Arial;margin:24px;color:#111} table{border-collapse:collapse;font-size:13px;margin:6px 0 18px} th,td{border:1px solid #ddd;padding:5px 9px} th{background:#f1f5f9} section{margin-bottom:26px}</style></head><body>" + ''.join(partes) + "</body></html>"
out_html = os.path.join(EDA_DIR, 'EDA_BraTS2024_GLI_reporte.html')
with open(out_html, 'w', encoding='utf-8') as f: f.write(html)
print('Reporte HTML guardado en', out_html)

In [ ]:
# === Empaquetar y descargar salidas del EDA ===
try:
    from google.colab import files; _COLAB = True
except Exception:
    _COLAB = False
nombres = ['EDA_volumenes_completo.csv','casos_demostrativos.csv','geometria_casos.csv',
           'parametros_registro.json','EDA_intensidad_otsu.csv','EDA_intensidad_stats.csv',
           'EDA_intensidad_seed.csv','EDA_BraTS2024_GLI_reporte.html']
zip_out = '/content/EDA_BraTS2024_GLI_salidas.zip' if EN_COLAB else 'EDA_salidas.zip'
with zipfile.ZipFile(zip_out, 'w', zipfile.ZIP_DEFLATED) as zf:
    for n in nombres:
        p = os.path.join(EDA_DIR, n)
        if os.path.exists(p): zf.write(p, n)
print('ZIP:', zip_out, round(os.path.getsize(zip_out)/1e6,2), 'MB')
if _COLAB: files.download(zip_out)